# Sparkov Fraud Detection: M3 Pseudo-Labeling with LightGBM

## Objective

This notebook extends the selected M2 model with pseudo-labeling. It uses only LightGBM with balanced class weights, expanding historical training data, weekly retraining, and a seven-day label verification delay.

M3 tunes two parameters:

- **Confidence threshold:** assign pseudo-fraud when predicted fraud probability is at least the threshold; assign pseudo-legitimate when it is at most one minus the threshold. Predictions between the two cutoffs are left out.
- **Reliability weight:** verified labels receive multiplier 1.0; selected pseudo-labels receive a lower multiplier.

The development period is the last 84 days of the original training split. The best pair is selected by development PR-AUC and evaluated once on the held-out validation split. The test split is not loaded or used.

### How the weights are applied

At each weekly retraining cutoff, balanced class weights are calculated from the verified training labels only. Each row receives:

`final sample weight = verified-label class weight × reliability multiplier`

The LightGBM estimator is then fitted with these per-row sample weights. This is equivalent to balanced class weighting while keeping pseudo-labels from changing the class-weight calculation. Actual class weights are recalculated each cutoff from its verified data; the 1 and 10 values in the example are illustrative.

| Transaction type | Class weight | Reliability multiplier | Final sample weight |
| --- | --- | --- | --- |
| Verified legitimate | balanced weight for class 0 | 1.0 | class 0 weight × 1.0 |
| Verified fraud | balanced weight for class 1 | 1.0 | class 1 weight × 1.0 |
| Pseudo-labeled legitimate | balanced weight for class 0 | selected value | class 0 weight × selected value |
| Pseudo-labeled fraud | balanced weight for class 1 | selected value | class 1 weight × selected value |

Grid Search:

`CONFIDENCE_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]`

`RELIABILITY_WEIGHTS = [0.10, 0.30, 0.50, 0.70, 0.90]`


In [1]:
from pathlib import Path
from itertools import product
import time
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/m3")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the processed feature table and the pre-approved modelling feature list.
df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")
with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)
feature_cols = feature_lists["sparkov_feature_cols"]

TIME_COL = "trans_date_trans_time"
TARGET_COL = "is_fraud"
SPLIT_COL = "sparkov_split"

# The test partition is excluded. Retain chronological train and validation only.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
m3_df = df.loc[df[SPLIT_COL].isin(["train", "validation"])].copy()
m3_df = m3_df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
del df

# Match M2: reconstruct selected frequency features using strictly earlier rows.
frequency_sources = {
    "merchant_freq": "merchant",
    "category_freq": "category",
    "state_freq": "state",
}
for feature, source in frequency_sources.items():
    if feature not in feature_cols:
        continue
    previous_rows = m3_df.groupby(source, dropna=False).cumcount()
    same_timestamp_previous = m3_df.groupby(
        [source, TIME_COL], dropna=False
    ).cumcount()
    m3_df[feature] = (previous_rows - same_timestamp_previous).astype("int64")

train_mask = m3_df[SPLIT_COL].eq("train")
X_train_schema = m3_df.loc[train_mask, feature_cols]
categorical_cols = X_train_schema.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Data shape:", m3_df.shape)
print("Features:", len(feature_cols))
print("Categorical features:", categorical_cols)
print("Fraud rates by split:")
display(m3_df.groupby(SPLIT_COL)[TARGET_COL].agg(["size", "mean"]))


Data shape: (1296675, 87)
Features: 62
Categorical features: ['category', 'gender', 'state']
Fraud rates by split:


,size,mean
sparkov_split,,
train,1037340,0.005753
validation,259335,0.005931


## Temporal setup

Only labels dated earlier than `cutoff − 7 days` are treated as verified and available for fitting. At each cutoff, transactions in the seven-day interval immediately before the cutoff form the unlabeled pool. A model trained only on verified data scores that pool; high-confidence predictions become pseudo-labels for the refit. The refit then scores the next seven-day block.


In [5]:
# Fixed M2 temporal configuration
WINDOW_DAYS = None                 # expanding history
RETRAIN_EVERY_DAYS = 7
LABEL_DELAY_DAYS = 7
CONFIDENCE_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
RELIABILITY_WEIGHTS = [0.10, 0.30, 0.50, 0.70, 0.90]


def make_lightgbm(class_weight="balanced"):
    """Best LightGBM tree settings from M1/M2."""
    return lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        objective="binary",
        class_weight=class_weight,
        is_unbalance=False,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )


def prepare_features(frame, categories=None):
    X = frame[feature_cols].copy()
    for col in categorical_cols:
        if categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=categories[col])
    return X


def verified_training_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select verified labels available at this cutoff; use expanding history."""
    cutoff = pd.Timestamp(cutoff)
    latest_eligible = cutoff - pd.Timedelta(days=label_delay_days)
    mask = frame[SPLIT_COL].isin(allowed_splits) & frame[TIME_COL].lt(latest_eligible)
    return frame.loc[mask].copy()


def pseudo_candidate_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select rows in the seven-day label-delay interval before the cutoff."""
    cutoff = pd.Timestamp(cutoff)
    start = cutoff - pd.Timedelta(days=label_delay_days)
    mask = (
        frame[SPLIT_COL].isin(allowed_splits)
        & frame[TIME_COL].ge(start)
        & frame[TIME_COL].lt(cutoff)
    )
    return frame.loc[mask].copy()


def make_cutoffs(start, end, frequency_days=RETRAIN_EVERY_DAYS):
    return pd.date_range(
        start=pd.Timestamp(start),
        end=pd.Timestamp(end),
        freq=f"{frequency_days}D",
        inclusive="left",
    )


def precompute_pseudo_scores(
    frame,
    start,
    end,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Cache one verified-only model's scores for each cutoff's delay window."""
    cache = {}
    cutoffs = make_cutoffs(start, end, frequency_days)

    for i, cutoff in enumerate(cutoffs, start=1):
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        candidates = pseudo_candidate_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        if verified.empty or verified[TARGET_COL].nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")
        if candidates.empty:
            cache[cutoff] = pd.DataFrame(
                columns=["row_index", "fraud_probability"]
            )
            continue

        X_verified = prepare_features(verified)
        categories = {col: X_verified[col].cat.categories for col in categorical_cols}
        X_candidates = prepare_features(candidates, categories=categories)
        model = make_lightgbm(class_weight="balanced")
        model.fit(X_verified, verified[TARGET_COL].astype(int))
        fraud_probability = model.predict_proba(X_candidates)[:, 1]

        cache[cutoff] = pd.DataFrame({
            "row_index": candidates.index.to_numpy(),
            "fraud_probability": fraud_probability,
        })
        print(
            f"Cached {i}/{len(cutoffs)} cutoffs: {cutoff} "
            f"({len(verified):,} verified; {len(candidates):,} unlabeled candidates)",
            flush=True,
        )
    return cache


def balanced_class_weight_map(y_verified):
    """Balanced class weights based only on verified labels at this cutoff."""
    classes = np.sort(pd.Series(y_verified).astype(int).unique())
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=pd.Series(y_verified).astype(int).to_numpy(),
    )
    return {int(label): float(weight) for label, weight in zip(classes, weights)}


def summarize_predictions(predictions):
    y_true = predictions["y_true"].astype(int)
    y_score = predictions["y_score"].astype(float)
    return {
        "transactions": len(predictions),
        "fraud_cases": int(y_true.sum()),
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score),
    }


In [6]:
def run_adaptive_lightgbm(
    frame,
    start,
    end,
    confidence_threshold=None,
    reliability_weight=0.0,
    pseudo_score_cache=None,
    allowed_splits=("train",),
    scoring_split="train",
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Weekly LightGBM retraining; None threshold gives the M2 baseline."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start >= end:
        raise ValueError("start must be earlier than end")
    if confidence_threshold is not None and not 0.5 <= confidence_threshold <= 1.0:
        raise ValueError("confidence_threshold must be at least 0.5 and at most 1")
    if not 0.0 <= reliability_weight <= 1.0:
        raise ValueError("reliability_weight must be between 0 and 1")

    prediction_blocks = []
    for cutoff in make_cutoffs(start, end, frequency_days):
        next_cutoff = min(cutoff + pd.Timedelta(days=frequency_days), end)
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        score_block = frame.loc[
            frame[SPLIT_COL].eq(scoring_split)
            & frame[TIME_COL].ge(cutoff)
            & frame[TIME_COL].lt(next_cutoff)
        ].copy()
        if score_block.empty:
            continue
        y_verified = verified[TARGET_COL].astype(int)
        if verified.empty or y_verified.nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")

        pseudo_rows = frame.iloc[0:0].copy()
        if confidence_threshold is not None:
            if pseudo_score_cache is None or cutoff not in pseudo_score_cache:
                raise KeyError(f"Missing pseudo-label scores for {cutoff}")
            scored_candidates = pseudo_score_cache[cutoff]
            if not scored_candidates.empty:
                p = scored_candidates["fraud_probability"].to_numpy()
                keep = (p >= confidence_threshold) | (p <= 1.0 - confidence_threshold)
                chosen = scored_candidates.loc[keep]
                if not chosen.empty:
                    pseudo_rows = frame.loc[chosen["row_index"].astype(int).to_numpy()].copy()
                    pseudo_rows["_m3_target"] = (
                        chosen["fraud_probability"].to_numpy() >= confidence_threshold
                    ).astype(int)

        verified_rows = verified.copy()
        verified_rows["_m3_target"] = y_verified.to_numpy()
        fit_rows = pd.concat([verified_rows, pseudo_rows], axis=0)
        y_fit = fit_rows["_m3_target"].astype(int)

        # Per-row sample weight = class weight from verified labels × reliability.
        class_weights = balanced_class_weight_map(y_verified)
        class_component = y_fit.map(class_weights).to_numpy(dtype=float)
        reliability_component = np.concatenate([
            np.ones(len(verified_rows), dtype=float),
            np.full(len(pseudo_rows), reliability_weight, dtype=float),
        ])
        fit_sample_weight = class_component * reliability_component

        X_fit = prepare_features(fit_rows)
        categories = {col: X_fit[col].cat.categories for col in categorical_cols}
        X_score = prepare_features(score_block, categories=categories)
        # Class balance is already represented in fit_sample_weight, avoiding double weighting.
        model = make_lightgbm(class_weight=None)
        model.fit(X_fit, y_fit, sample_weight=fit_sample_weight)
        scores = model.predict_proba(X_score)[:, 1]

        prediction_blocks.append(pd.DataFrame({
            "row_index": score_block.index.to_numpy(),
            "timestamp": score_block[TIME_COL].to_numpy(),
            "y_true": score_block[TARGET_COL].astype(int).to_numpy(),
            "y_score": scores,
            "cutoff": cutoff,
            "confidence_threshold": confidence_threshold,
            "reliability_weight": reliability_weight,
            "verified_rows": len(verified_rows),
            "pseudo_rows": len(pseudo_rows),
            "class_weight_legitimate": class_weights[0],
            "class_weight_fraud": class_weights[1],
        }))

    if not prediction_blocks:
        raise ValueError("No scoring rows found")
    predictions = pd.concat(prediction_blocks, ignore_index=True)
    if predictions["row_index"].duplicated().any():
        raise AssertionError("Duplicate scored rows")
    return predictions


## Development tuning

The grid search uses only transactions in the original training partition. Candidate score probabilities are cached once per weekly cutoff, then reused for all parameter combinations. Each combination refits the final LightGBM at each cutoff using its own selected pseudo-labels and reliability multiplier.


In [7]:
# Development window: the final 84 days of the original training period.
train_end = m3_df.loc[m3_df[SPLIT_COL].eq("train"), TIME_COL].max()
DEV_END = train_end + pd.Timedelta(microseconds=1)
DEV_START = DEV_END - pd.Timedelta(days=84)
print("Development start:", DEV_START)
print("Development end:  ", DEV_END)

DEV_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df,
    start=DEV_START,
    end=DEV_END,
    allowed_splits=("train",),
)


def config_id(threshold, reliability):
    return (
        f"threshold_{int(round(threshold * 100))}_"
        f"reliability_{int(round(reliability * 100))}"
    )


def tune_m3(frame, overwrite=False):
    results_path = RESULTS_DIR / "m3_development_grid.csv"
    rows = []
    configs = list(product(CONFIDENCE_THRESHOLDS, RELIABILITY_WEIGHTS))
    reference_rows = None

    for i, (threshold, reliability) in enumerate(configs, start=1):
        name = config_id(threshold, reliability)
        prediction_path = RESULTS_DIR / f"development_{name}.csv"
        print(f"\n[{i}/{len(configs)}] {name}", flush=True)
        started = time.perf_counter()

        if prediction_path.exists() and not overwrite:
            pred = pd.read_csv(prediction_path, parse_dates=["timestamp", "cutoff"])
        else:
            pred = run_adaptive_lightgbm(
                frame=frame,
                start=DEV_START,
                end=DEV_END,
                confidence_threshold=threshold,
                reliability_weight=reliability,
                pseudo_score_cache=DEV_PSEUDO_CACHE,
                allowed_splits=("train",),
                scoring_split="train",
            )
            pred.to_csv(prediction_path, index=False)

        metrics = summarize_predictions(pred)
        row = {
            "config_id": name,
            "confidence_threshold": threshold,
            "reliability_weight": reliability,
            **metrics,
            "mean_pseudo_rows_per_retrain": pred["pseudo_rows"].mean(),
        }
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)

        scored_rows = set(pred["row_index"].astype(int))
        if reference_rows is None:
            reference_rows = scored_rows
        else:
            assert scored_rows == reference_rows, f"Scored rows differ for {name}"

        print(
            f"PR-AUC={metrics['pr_auc']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | "
            f"Mean pseudo rows/retrain={row['mean_pseudo_rows_per_retrain']:.0f} | "
            f"Elapsed={time.perf_counter() - started:.1f}s",
            flush=True,
        )

    return pd.DataFrame(rows).sort_values(
        ["pr_auc", "roc_auc"], ascending=[False, False]
    ).reset_index(drop=True)


development_results = tune_m3(m3_df)
display(development_results)
best_config = development_results.iloc[0].to_dict()
print("Selected configuration:")
display(pd.DataFrame([best_config]))
pd.DataFrame([best_config]).to_csv(
    RESULTS_DIR / "m3_best_config.csv", index=False
)


Development start: 2019-12-13 07:15:17.000001
Development end:   2020-03-06 07:15:17.000001
Cached 1/12 cutoffs: 2019-12-13 07:15:17.000001 (807,061 verified; 31,649 unlabeled candidates)
Cached 2/12 cutoffs: 2019-12-20 07:15:17.000001 (838,710 verified; 31,465 unlabeled candidates)
Cached 3/12 cutoffs: 2019-12-27 07:15:17.000001 (870,175 verified; 31,031 unlabeled candidates)
Cached 4/12 cutoffs: 2020-01-03 07:15:17.000001 (901,206 verified; 26,625 unlabeled candidates)
Cached 5/12 cutoffs: 2020-01-10 07:15:17.000001 (927,831 verified; 12,021 unlabeled candidates)
Cached 6/12 cutoffs: 2020-01-17 07:15:17.000001 (939,852 verified; 12,111 unlabeled candidates)
Cached 7/12 cutoffs: 2020-01-24 07:15:17.000001 (951,963 verified; 12,049 unlabeled candidates)
Cached 8/12 cutoffs: 2020-01-31 07:15:17.000001 (964,012 verified; 11,960 unlabeled candidates)
Cached 9/12 cutoffs: 2020-02-07 07:15:17.000001 (975,972 verified; 11,942 unlabeled candidates)
Cached 10/12 cutoffs: 2020-02-14 07:15:17.00

,config_id,confidence_threshold,reliability_weight,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,threshold_60_reliability_10,0.60,0.1,198630,1107,0.986425,0.999798,21576.654005
1,threshold_95_reliability_70,0.95,0.7,198630,1107,0.986408,0.999792,21487.222771
2,threshold_60_reliability_90,0.60,0.9,198630,1107,0.986365,0.999774,21576.654005
3,threshold_90_reliability_90,0.90,0.9,198630,1107,0.986246,0.999831,21523.403086
4,threshold_70_reliability_30,0.70,0.3,198630,1107,0.986190,0.999788,21565.675104
5,threshold_90_reliability_70,0.90,0.7,198630,1107,0.986122,0.999818,21523.403086
6,threshold_95_reliability_30,0.95,0.3,198630,1107,0.986041,0.999792,21487.222771
7,threshold_50_reliability_10,0.50,0.1,198630,1107,0.985972,0.999782,21586.347959
8,threshold_95_reliability_90,0.95,0.9,198630,1107,0.985890,0.999800,21487.222771
9,threshold_95_reliability_50,0.95,0.5,198630,1107,0.985856,0.999799,21487.222771


Selected configuration:


,config_id,confidence_threshold,reliability_weight,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,threshold_60_reliability_10,0.6,0.1,198630,1107,0.986425,0.999798,21576.654005


## Held-out validation

After selecting the parameter pair using development PR-AUC, compare it against the M2-style LightGBM baseline on the validation period. Both use expanding history, weekly retraining, the same seven-day label delay, and the same chronological validation rows. The baseline uses verified labels only; M3 adds high-confidence pseudo-labels.


In [8]:
validation_rows = m3_df.loc[m3_df[SPLIT_COL].eq("validation")]
VAL_START = validation_rows[TIME_COL].min()
VAL_END = validation_rows[TIME_COL].max() + pd.Timedelta(microseconds=1)
print("Validation rows:", len(validation_rows))
print("Validation start:", VAL_START)
print("Validation end:  ", VAL_END)

VAL_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    allowed_splits=("train", "validation"),
)

# M2 reference: same temporal setup, but no pseudo-labeled observations.
m2_baseline_predictions = run_adaptive_lightgbm(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    confidence_threshold=None,
    reliability_weight=0.0,
    pseudo_score_cache=None,
    allowed_splits=("train", "validation"),
    scoring_split="validation",
)

m3_validation_predictions = run_adaptive_lightgbm(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    confidence_threshold=float(best_config["confidence_threshold"]),
    reliability_weight=float(best_config["reliability_weight"]),
    pseudo_score_cache=VAL_PSEUDO_CACHE,
    allowed_splits=("train", "validation"),
    scoring_split="validation",
)

expected_rows = set(m3_df.index[m3_df[SPLIT_COL].eq("validation")])
for label, pred in [
    ("M2 baseline", m2_baseline_predictions),
    ("M3 pseudo-labeling", m3_validation_predictions),
]:
    assert len(pred) == len(expected_rows)
    assert set(pred["row_index"].astype(int)) == expected_rows

m2_metrics = summarize_predictions(m2_baseline_predictions)
m3_metrics = summarize_predictions(m3_validation_predictions)
validation_comparison = pd.DataFrame([
    {
        "model": "LightGBM",
        "strategy": "M2 balanced, verified labels only",
        "confidence_threshold": np.nan,
        "reliability_weight": np.nan,
        **m2_metrics,
    },
    {
        "model": "LightGBM",
        "strategy": "M3 balanced, pseudo-labeling",
        "confidence_threshold": float(best_config["confidence_threshold"]),
        "reliability_weight": float(best_config["reliability_weight"]),
        **m3_metrics,
        "mean_pseudo_rows_per_retrain": m3_validation_predictions["pseudo_rows"].mean(),
    },
])
display(validation_comparison)
validation_comparison.to_csv(RESULTS_DIR / "m3_validation_comparison.csv", index=False)
m3_validation_predictions.to_csv(
    RESULTS_DIR / "m3_validation_predictions.csv", index=False
)


Validation rows: 259335
Validation start: 2020-03-06 07:16:43
Validation end:   2020-06-21 12:13:37.000001
Cached 1/16 cutoffs: 2020-03-06 07:16:43 (1,023,767 verified; 13,573 unlabeled candidates)
Cached 2/16 cutoffs: 2020-03-13 07:16:43 (1,037,340 verified; 16,114 unlabeled candidates)
Cached 3/16 cutoffs: 2020-03-20 07:16:43 (1,053,454 verified; 16,087 unlabeled candidates)
Cached 4/16 cutoffs: 2020-03-27 07:16:43 (1,069,541 verified; 15,950 unlabeled candidates)
Cached 5/16 cutoffs: 2020-04-03 07:16:43 (1,085,491 verified; 15,855 unlabeled candidates)
Cached 6/16 cutoffs: 2020-04-10 07:16:43 (1,101,346 verified; 16,057 unlabeled candidates)
Cached 7/16 cutoffs: 2020-04-17 07:16:43 (1,117,403 verified; 15,984 unlabeled candidates)
Cached 8/16 cutoffs: 2020-04-24 07:16:43 (1,133,387 verified; 15,766 unlabeled candidates)
Cached 9/16 cutoffs: 2020-05-01 07:16:43 (1,149,153 verified; 15,872 unlabeled candidates)
Cached 10/16 cutoffs: 2020-05-08 07:16:43 (1,165,025 verified; 15,677 unla

,model,strategy,confidence_threshold,reliability_weight,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,LightGBM,"M2 balanced, verified labels only",NaN,NaN,259335,1538,0.985819,0.999561,NaN
1,LightGBM,"M3 balanced, pseudo-labeling",0.6,0.1,259335,1538,0.985842,0.999666,16697.03175


## Reading the outcome

Use the held-out validation comparison to judge whether pseudo-labeling helped. Development results select the threshold and reliability multiplier; validation is an evaluation, not another tuning set. A threshold that selects very few rows may act much like M2. A lower threshold selects more rows but can add incorrect labels, so inspect the pseudo-row counts as well as PR-AUC.

The reported class weights are computed from verified labels at each cutoff. For each pseudo-labeled row, the reliability multiplier is then applied to the class weight for its predicted class. For example, if the verified-only fraud weight at a cutoff is 10 and the selected reliability multiplier is 0.3, a pseudo-fraud row has sample weight 3. Verified fraud retains weight 10. In the actual dataset, the dynamically balanced class weight may be quite different from 10.
